# CSE 547 Final Project Report 2 - Source Notebook

This notebook is self-contained: it defines the models, training loops, plotting, prediction mining, report generation, and video outline generation used for the final submission.


In [ ]:
REUSE_CHECKPOINTS = True
SMOKE_TEST = False


## Implementation
Run this cell to define all project code.


In [ ]:
"""
Report 2 pipeline for the CSE 547 final project.

This script completes the remaining project requirements:
  - Part 3: RGB VGG16 transfer learning with 3 freeze settings.
  - Part 4: IR convolutional autoencoder features with 12 classifier options.
  - Part 5: misclassification mining and RGB-vs-IR comparison.
  - Part 6: bounded refinement and final model selection.
  - Final 5-page PDF report, final notebook, and video outline.

The final notebook generated by this script embeds the implementation source
directly, so it is independently runnable.
"""

from __future__ import annotations

import argparse
import ast
import copy
import csv
import json
import math
import os
import random
import sys
import textwrap
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tv_models
from PIL import Image
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
from reportlab.platypus import Image as RLImage
from reportlab.lib.utils import ImageReader
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms


try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd().resolve()
FIGURES_DIR = ROOT / "figures"
CHECKPOINT_DIR = ROOT / "checkpoints"
MANIFEST_DIR = ROOT / "manifests"
REPORT_PDF = ROOT / "CSE547_FinalProject_Report2_Fuentes.pdf"
FINAL_NOTEBOOK = ROOT / "CSE547_FinalProject_Report2_Fuentes.ipynb"
VIDEO_OUTLINE = ROOT / "REPORT2_VIDEO_OUTLINE.md"

RGB_MANIFEST = MANIFEST_DIR / "rgb_manifest.csv"
IR_MANIFEST = MANIFEST_DIR / "ir_manifest.csv"
PAIRED_SCENES = MANIFEST_DIR / "paired_scene_eval.csv"
INDEX_JSON = ROOT / "index.json"

CLASSES = {
    "bike": 0,
    "bus": 1,
    "car": 2,
    "person": 3,
    "sign": 4,
    "motor": 5,
    "light": 6,
    "truck": 7,
}
ID_TO_CLASS = {v: k for k, v in CLASSES.items()}
NUM_CLASSES = len(CLASSES)

RGB_PART1_BASELINE = {
    "label": "RGB Part 1 Arch D",
    "modality": "RGB",
    "source": "part1_rgb_archD",
    "best_val_acc": 0.9201,
    "best_val_f1": 0.9185,
    "best_train_acc": 0.9999,
    "best_train_f1": 0.9999,
    "checkpoint": str(CHECKPOINT_DIR / "part1_rgb_archD.pt"),
    "kind": "cnn",
    "arch": "D",
}

IR_PART1_BASELINE = {
    "label": "IR Part 1 Arch D",
    "modality": "IR",
    "source": "part1_ir_archD",
    "best_val_acc": 0.8880,
    "best_val_f1": 0.8792,
    "best_train_acc": 0.9766,
    "best_train_f1": 0.9772,
    "checkpoint": str(CHECKPOINT_DIR / "part1_ir_archD.pt"),
    "kind": "cnn",
    "arch": "D",
}

IR_REGULARIZED_REFERENCE = {
    "label": "IR CNN Arch D + Aug L2",
    "modality": "IR",
    "source": "part2_ir_aug_2",
    "best_val_acc": 0.9263,
    "best_val_f1": 0.9227,
    "best_train_acc": 0.0,
    "best_train_f1": 0.0,
    "checkpoint": str(CHECKPOINT_DIR / "part2_ir_aug_2.pt"),
    "kind": "cnn",
    "arch": "D",
}

AE_CONFIGS = {
    "AE1": {"filters": [16, 32], "bottleneck_dim": 64},
    "AE2": {"filters": [32, 64], "bottleneck_dim": 128},
    "AE3": {"filters": [16, 32, 64], "bottleneck_dim": 64},
    "AE4": {"filters": [32, 64, 128], "bottleneck_dim": 128},
    "AE5": {"filters": [16, 32, 64, 128], "bottleneck_dim": 256},
    "AE6": {"filters": [32, 64, 128, 256], "bottleneck_dim": 512},
}

AE_REG_CONFIGS = {
    "R1": {"dropout": 0.20, "weight_decay": 1e-5},
    "R2": {"dropout": 0.50, "weight_decay": 1e-4},
}

VGG_FREEZE_CONFIGS = {
    "F1": {"freeze_level": 1, "description": "freeze all conv blocks"},
    "F2": {"freeze_level": 2, "description": "train block 5 + head"},
    "F3": {"freeze_level": 3, "description": "train blocks 4-5 + head"},
}

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")


def ensure_dirs() -> None:
    FIGURES_DIR.mkdir(exist_ok=True)
    CHECKPOINT_DIR.mkdir(exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def read_json(path: Path) -> Any:
    with path.open(encoding="utf-8") as f:
        return json.load(f)


def checkpoint_exists(path: str | Path) -> bool:
    p = Path(path)
    return p.exists() and p.stat().st_size > 0


def load_state_dict(path: str | Path) -> dict[str, Any]:
    return torch.load(path, map_location="cpu")


def save_state_dict(model: nn.Module, path: str | Path) -> None:
    Path(path).parent.mkdir(exist_ok=True)
    torch.save(model.state_dict(), path)


def to_jsonable(history: dict[str, Any]) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for k, v in history.items():
        if isinstance(v, list):
            out[k] = [float(x) for x in v]
        elif isinstance(v, (np.floating, np.integer)):
            out[k] = float(v)
        elif isinstance(v, Path):
            out[k] = str(v)
        else:
            out[k] = v
    return out


def stratified_subset(df: pd.DataFrame, per_class: int) -> pd.DataFrame:
    if per_class <= 0:
        return df
    pieces = []
    for _, group in df.groupby("label_id", sort=True):
        pieces.append(group.head(per_class))
    return pd.concat(pieces, ignore_index=True)


class ManifestImageDataset(Dataset):
    def __init__(self, manifest_df: pd.DataFrame, transform: transforms.Compose) -> None:
        self.df = manifest_df.reset_index(drop=True)
        self.paths = self.df["patch_path"].tolist()
        self.labels = self.df["label_id"].astype(int).tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        with Image.open(self.paths[idx]) as src:
            img = src.convert("RGB")
        return self.transform(img), self.labels[idx]


def worker_init(_: int) -> None:
    torch.set_num_threads(1)


def make_loaders(
    manifest_path: Path,
    train_transform: transforms.Compose,
    val_transform: transforms.Compose,
    batch_size: int,
    smoke_test: bool = False,
    smoke_per_class_train: int = 32,
    smoke_per_class_val: int = 16,
    shuffle_train: bool = True,
    num_workers: int | None = None,
) -> tuple[DataLoader, DataLoader, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(manifest_path)
    train_df = pd.DataFrame(df[df["split"] == "train"])
    val_df = pd.DataFrame(df[df["split"] == "val"])
    if smoke_test:
        train_df = stratified_subset(train_df, smoke_per_class_train)
        val_df = stratified_subset(val_df, smoke_per_class_val)
    if num_workers is None:
        in_notebook = "ipykernel" in sys.modules
        if smoke_test or (os.name == "nt" and in_notebook):
            num_workers = 0
        else:
            num_workers = min(8, os.cpu_count() or 4)
    kwargs: dict[str, Any] = {
        "num_workers": num_workers,
        "pin_memory": DEVICE.type == "cuda",
    }
    if num_workers > 0:
        kwargs["prefetch_factor"] = 2
        kwargs["persistent_workers"] = True
        kwargs["worker_init_fn"] = worker_init
    train_ds = ManifestImageDataset(train_df, train_transform)
    val_ds = ManifestImageDataset(val_df, val_transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle_train, **kwargs)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, **kwargs)
    return train_loader, val_loader, train_df, val_df


def imagenet_train_transform(augment: bool = False) -> transforms.Compose:
    if augment:
        steps: list[Any] = [
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05),
        ]
    else:
        steps = [transforms.Resize((224, 224))]
    steps.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    return transforms.Compose(steps)


def imagenet_val_transform() -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


def base_patch_transform(img_size: int = 64, augment: bool = False) -> transforms.Compose:
    steps: list[Any] = [transforms.Resize((img_size, img_size))]
    if augment:
        steps.extend(
            [
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            ]
        )
    steps.append(transforms.ToTensor())
    return transforms.Compose(steps)


def amp_context() -> tuple[str, torch.dtype, bool]:
    if DEVICE.type == "cuda":
        return "cuda", torch.float16, True
    return "cpu", torch.bfloat16, False


def accuracy_f1(labels: list[int] | np.ndarray, preds: list[int] | np.ndarray) -> tuple[float, float]:
    return float(accuracy_score(labels, preds)), float(
        f1_score(labels, preds, average="weighted", zero_division=0)
    )


def evaluate_classifier(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module | None = None,
) -> tuple[float, float, float]:
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    amp_device, amp_dtype, amp_enabled = amp_context()
    model.eval()
    running_loss = 0.0
    preds_all: list[int] = []
    labels_all: list[int] = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=amp_device, dtype=amp_dtype, enabled=amp_enabled):
                logits = model(images)
                loss = criterion(logits, labels)
            running_loss += float(loss.item()) * images.size(0)
            preds_all.extend(logits.argmax(1).detach().cpu().tolist())
            labels_all.extend(labels.detach().cpu().tolist())
    acc, f1 = accuracy_f1(labels_all, preds_all)
    return running_loss / len(loader.dataset), acc, f1


def train_image_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    checkpoint_path: Path,
    epochs: int,
    min_epochs: int,
    patience: int,
    lr: float = 3e-4,
    weight_decay: float = 0.0,
    param_groups: list[dict[str, Any]] | None = None,
    class_weights: torch.Tensor | None = None,
) -> dict[str, Any]:
    model = model.to(DEVICE)
    amp_device, amp_dtype, amp_enabled = amp_context()
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE) if class_weights is not None else None)
    optimizer = torch.optim.Adam(param_groups if param_groups is not None else model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    history: dict[str, Any] = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_f1": [],
        "val_f1": [],
        "lr_history": [],
    }
    best_state = None
    best_val_f1 = -1.0
    best_epoch = 0
    stale = 0
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        preds_all: list[int] = []
        labels_all: list[int] = []
        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=amp_device, dtype=amp_dtype, enabled=amp_enabled):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.item()) * images.size(0)
            preds_all.extend(logits.argmax(1).detach().cpu().tolist())
            labels_all.extend(labels.detach().cpu().tolist())
        train_acc, train_f1 = accuracy_f1(labels_all, preds_all)
        val_loss, val_acc, val_f1 = evaluate_classifier(model, val_loader, criterion)
        train_loss = running / len(train_loader.dataset)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)
        history["lr_history"].append(float(optimizer.param_groups[0]["lr"]))
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 5 == 0:
            print(
                f"  epoch {epoch:02d}/{epochs} loss {train_loss:.4f}/{val_loss:.4f} "
                f"acc {train_acc:.4f}/{val_acc:.4f} f1 {train_f1:.4f}/{val_f1:.4f}"
            )
        if epoch >= min_epochs and stale >= patience:
            print(f"  early stop at epoch {epoch}; best epoch {best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
        save_state_dict(model, checkpoint_path)
    val_loss, val_acc, val_f1 = evaluate_classifier(model, val_loader, criterion)
    train_loss, train_acc, train_f1 = evaluate_classifier(model, train_loader, criterion)
    history.update(
        {
            "best_epoch": best_epoch,
            "best_train_acc": train_acc,
            "best_val_acc": val_acc,
            "best_train_f1": train_f1,
            "best_val_f1": val_f1,
            "best_train_loss": train_loss,
            "best_val_loss": val_loss,
        }
    )
    return to_jsonable(history)


def predict_image_classifier(model: nn.Module, loader: DataLoader, val_df: pd.DataFrame) -> pd.DataFrame:
    amp_device, amp_dtype, amp_enabled = amp_context()
    model = model.to(DEVICE)
    model.eval()
    preds: list[int] = []
    labels_out: list[int] = []
    probs: list[float] = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=amp_device, dtype=amp_dtype, enabled=amp_enabled):
                logits = model(images)
                p = torch.softmax(logits, dim=1)
            pred = p.argmax(1)
            preds.extend(pred.cpu().tolist())
            labels_out.extend(labels.cpu().tolist())
            probs.extend(p.max(1).values.cpu().tolist())
    df = val_df.reset_index(drop=True).copy()
    df["true_id"] = labels_out
    df["pred_id"] = preds
    df["pred_class"] = [ID_TO_CLASS[int(x)] for x in preds]
    df["true_class"] = [ID_TO_CLASS[int(x)] for x in labels_out]
    df["confidence"] = probs
    df["correct"] = df["true_id"] == df["pred_id"]
    return df


def _conv_block(in_ch: int, out_ch: int, use_bn: bool = False) -> nn.Sequential:
    layers: list[nn.Module] = [nn.Conv2d(in_ch, out_ch, 3, padding=1)]
    if use_bn:
        layers.append(nn.BatchNorm2d(out_ch))
    layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)


class CNNArchD(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES) -> None:
        super().__init__()
        self.features = nn.Sequential(
            _conv_block(3, 64),
            _conv_block(64, 64),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(64),
            _conv_block(64, 128),
            _conv_block(128, 128),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(128),
            _conv_block(128, 256),
            _conv_block(256, 256),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(256),
            _conv_block(256, 512),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(self.gap(x).flatten(1))


VGG16_BLOCKS = {
    "block1": (0, 4),
    "block2": (5, 9),
    "block3": (10, 16),
    "block4": (17, 23),
    "block5": (24, 30),
}


class VGG16Transfer(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, freeze_level: int = 1) -> None:
        super().__init__()
        weights = tv_models.VGG16_Weights.IMAGENET1K_V1
        vgg = tv_models.vgg16(weights=weights)
        self.features = vgg.features
        self._apply_freeze(freeze_level)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def _apply_freeze(self, level: int) -> None:
        for param in self.features.parameters():
            param.requires_grad = False
        if level >= 2:
            start, end = VGG16_BLOCKS["block5"]
            for idx in range(start, end + 1):
                for param in self.features[idx].parameters():
                    param.requires_grad = True
        if level >= 3:
            start, end = VGG16_BLOCKS["block4"]
            for idx in range(start, end + 1):
                for param in self.features[idx].parameters():
                    param.requires_grad = True

    def get_param_groups(self, lr_pretrained: float = 1e-5, lr_head: float = 3e-4) -> list[dict[str, Any]]:
        pretrained_params = [p for p in self.features.parameters() if p.requires_grad]
        groups: list[dict[str, Any]] = [{"params": self.classifier.parameters(), "lr": lr_head}]
        if pretrained_params:
            groups.insert(0, {"params": pretrained_params, "lr": lr_pretrained})
        return groups

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


class ConvEncoder(nn.Module):
    def __init__(self, filters: list[int], bottleneck_dim: int) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        in_ch = 3
        for out_ch in filters:
            layers.extend([nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2)])
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        spatial = 64 // (2 ** len(filters))
        self.flat_size = in_ch * spatial * spatial
        self.fc = nn.Linear(self.flat_size, bottleneck_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(self.conv(x).flatten(1))


class ConvDecoder(nn.Module):
    def __init__(self, filters: list[int], bottleneck_dim: int) -> None:
        super().__init__()
        rev = list(reversed(filters))
        spatial = 64 // (2 ** len(filters))
        self.first_ch = rev[0]
        self.spatial = spatial
        self.fc = nn.Linear(bottleneck_dim, self.first_ch * spatial * spatial)
        layers: list[nn.Module] = []
        in_ch = rev[0]
        for out_ch in rev[1:]:
            layers.extend([nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2), nn.ReLU(inplace=True)])
            in_ch = out_ch
        layers.extend([nn.ConvTranspose2d(in_ch, 3, 2, stride=2), nn.Sigmoid()])
        self.deconv = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.fc(z).view(-1, self.first_ch, self.spatial, self.spatial)
        return self.deconv(x)


class ConvAutoEncoder(nn.Module):
    def __init__(self, filters: list[int], bottleneck_dim: int) -> None:
        super().__init__()
        self.encoder = ConvEncoder(filters, bottleneck_dim)
        self.decoder = ConvDecoder(filters, bottleneck_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)


class DenseClassifier(nn.Module):
    def __init__(self, input_dim: int, dropout: float, num_classes: int = NUM_CLASSES) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_autoencoder(
    model: ConvAutoEncoder,
    train_loader: DataLoader,
    val_loader: DataLoader,
    checkpoint_path: Path,
    epochs: int,
    min_epochs: int,
    patience: int,
    lr: float = 1e-3,
) -> dict[str, Any]:
    model = model.to(DEVICE)
    amp_device, amp_dtype, amp_enabled = amp_context()
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    history: dict[str, Any] = {"train_loss": [], "val_loss": [], "lr_history": []}
    best_state = None
    best_loss = math.inf
    best_epoch = 0
    stale = 0
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for images, _ in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=amp_device, dtype=amp_dtype, enabled=amp_enabled):
                recon = model(images)
                loss = criterion(recon, images)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.item()) * images.size(0)
        train_loss = running / len(train_loader.dataset)
        val_loss = evaluate_autoencoder(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr_history"].append(float(optimizer.param_groups[0]["lr"]))
        scheduler.step(val_loss)
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 5 == 0:
            print(f"  AE epoch {epoch:02d}/{epochs} mse {train_loss:.5f}/{val_loss:.5f}")
        if epoch >= min_epochs and stale >= patience:
            print(f"  AE early stop at epoch {epoch}; best epoch {best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
        save_state_dict(model, checkpoint_path)
    history["best_epoch"] = best_epoch
    history["best_train_loss"] = evaluate_autoencoder(model, train_loader, criterion)
    history["best_val_loss"] = evaluate_autoencoder(model, val_loader, criterion)
    return to_jsonable(history)


def evaluate_autoencoder(model: ConvAutoEncoder, loader: DataLoader, criterion: nn.Module) -> float:
    amp_device, amp_dtype, amp_enabled = amp_context()
    model.eval()
    running = 0.0
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=amp_device, dtype=amp_dtype, enabled=amp_enabled):
                recon = model(images)
                loss = criterion(recon, images)
            running += float(loss.item()) * images.size(0)
    return running / len(loader.dataset)


def extract_features(ae: ConvAutoEncoder, loader: DataLoader) -> tuple[torch.Tensor, torch.Tensor]:
    amp_device, amp_dtype, amp_enabled = amp_context()
    ae = ae.to(DEVICE)
    ae.eval()
    feats: list[torch.Tensor] = []
    labels_out: list[torch.Tensor] = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=amp_device, dtype=amp_dtype, enabled=amp_enabled):
                z = ae.encode(images)
            feats.append(z.float().cpu())
            labels_out.append(labels.cpu())
    return torch.cat(feats, dim=0), torch.cat(labels_out, dim=0)


def train_feature_classifier(
    model: DenseClassifier,
    train_ds: TensorDataset,
    val_ds: TensorDataset,
    checkpoint_path: Path,
    epochs: int,
    min_epochs: int,
    patience: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
) -> dict[str, Any]:
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=DEVICE.type == "cuda")
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=DEVICE.type == "cuda")
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
    history: dict[str, Any] = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}
    best_state = None
    best_val_f1 = -1.0
    best_epoch = 0
    stale = 0
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        preds_all: list[int] = []
        labels_all: list[int] = []
        for feats, labels in train_loader:
            feats = feats.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(feats)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running += float(loss.item()) * feats.size(0)
            preds_all.extend(logits.argmax(1).detach().cpu().tolist())
            labels_all.extend(labels.detach().cpu().tolist())
        train_loss = running / len(train_ds)
        train_acc, train_f1 = accuracy_f1(labels_all, preds_all)
        val_loss, val_acc, val_f1 = evaluate_feature_classifier(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"  classifier epoch {epoch:02d}/{epochs} acc {train_acc:.4f}/{val_acc:.4f} f1 {train_f1:.4f}/{val_f1:.4f}")
        if epoch >= min_epochs and stale >= patience:
            print(f"  classifier early stop at epoch {epoch}; best epoch {best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
        save_state_dict(model, checkpoint_path)
    train_loss, train_acc, train_f1 = evaluate_feature_classifier(model, train_loader, criterion)
    val_loss, val_acc, val_f1 = evaluate_feature_classifier(model, val_loader, criterion)
    history.update(
        {
            "best_epoch": best_epoch,
            "best_train_loss": train_loss,
            "best_val_loss": val_loss,
            "best_train_acc": train_acc,
            "best_val_acc": val_acc,
            "best_train_f1": train_f1,
            "best_val_f1": val_f1,
        }
    )
    return to_jsonable(history)


def evaluate_feature_classifier(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> tuple[float, float, float]:
    model.eval()
    running = 0.0
    preds_all: list[int] = []
    labels_all: list[int] = []
    with torch.no_grad():
        for feats, labels in loader:
            feats = feats.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(feats)
            loss = criterion(logits, labels)
            running += float(loss.item()) * feats.size(0)
            preds_all.extend(logits.argmax(1).cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    acc, f1 = accuracy_f1(labels_all, preds_all)
    return running / len(loader.dataset), acc, f1


def predict_feature_classifier(model: nn.Module, features: torch.Tensor, labels: torch.Tensor, val_df: pd.DataFrame, batch_size: int = 2048) -> pd.DataFrame:
    ds = TensorDataset(features, labels)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model = model.to(DEVICE)
    model.eval()
    preds: list[int] = []
    probs: list[float] = []
    with torch.no_grad():
        for feats, _ in loader:
            logits = model(feats.to(DEVICE))
            p = torch.softmax(logits, dim=1)
            preds.extend(p.argmax(1).cpu().tolist())
            probs.extend(p.max(1).values.cpu().tolist())
    df = val_df.reset_index(drop=True).copy()
    df["true_id"] = labels.cpu().tolist()
    df["pred_id"] = preds
    df["pred_class"] = [ID_TO_CLASS[int(x)] for x in preds]
    df["true_class"] = [ID_TO_CLASS[int(x)] for x in labels.cpu().tolist()]
    df["confidence"] = probs
    df["correct"] = df["true_id"] == df["pred_id"]
    return df


def plot_4panel(results: list[dict[str, Any]], labels: list[str], title: str, x_label: str, save_path: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.5))
    x = np.arange(len(results))
    metrics = [
        ("best_train_acc", "Training Accuracy", "#1f77b4"),
        ("best_val_acc", "Validation Accuracy", "#d62728"),
        ("best_train_f1", "Training Weighted F1", "#2ca02c"),
        ("best_val_f1", "Validation Weighted F1", "#9467bd"),
    ]
    for ax, (key, ylabel, color) in zip(axes.flat, metrics):
        values = [float(r.get(key, 0.0)) for r in results]
        ax.plot(x, values, marker="o", linewidth=2.2, color=color, label=ylabel)
        for xi, yi in zip(x, values):
            ax.text(xi, yi + 0.006, f"{yi:.3f}", ha="center", va="bottom", fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35 if len(labels) > 6 else 0, ha="right" if len(labels) > 6 else "center")
        ax.set_xlabel(x_label)
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="lower right", fontsize=8)
    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    save_path.parent.mkdir(exist_ok=True)
    fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {save_path}")


def plot_misclassified_grid(items: list[dict[str, Any]], title: str, save_path: Path) -> None:
    if not items:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, "No qualifying samples found", ha="center", va="center", fontsize=16)
        ax.axis("off")
        fig.suptitle(title, fontsize=14, fontweight="bold")
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return
    cols = 4
    rows = math.ceil(len(items) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.1, rows * 3.2))
    axes_arr = np.array(axes).reshape(rows, cols)
    for ax in axes_arr.flat:
        ax.axis("off")
    for idx, item in enumerate(items):
        ax = axes_arr.flat[idx]
        try:
            img = Image.open(item["patch_path"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "image missing", ha="center", va="center")
        ax.set_title(f"{item['true_class']} -> {item['pred_class']}", fontsize=9, color="#b00020")
        ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {save_path}")


def plot_sensor_pairs(pairs: list[dict[str, Any]], title: str, save_path: Path) -> None:
    if not pairs:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, "No paired examples matched the selection rule", ha="center", va="center", fontsize=14)
        ax.axis("off")
        fig.suptitle(title, fontsize=14, fontweight="bold")
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return
    pairs = pairs[:4]
    fig, axes = plt.subplots(len(pairs), 2, figsize=(7.5, 3.2 * len(pairs)))
    axes_arr = np.array(axes).reshape(len(pairs), 2)
    for row_idx, pair in enumerate(pairs):
        for col_idx, sensor in enumerate(["rgb", "ir"]):
            ax = axes_arr[row_idx, col_idx]
            row = pair[sensor]
            try:
                img = Image.open(row["patch_path"]).convert("RGB")
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, "image missing", ha="center", va="center")
            verdict = "OK" if bool(row["correct"]) else "WRONG"
            color = "#167a35" if bool(row["correct"]) else "#b00020"
            ax.set_title(
                f"{sensor.upper()} {verdict}: {row['true_class']} -> {row['pred_class']}",
                fontsize=9,
                color=color,
            )
            ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {save_path}")


def run_part3_vgg16(reuse_checkpoints: bool = True, smoke_test: bool = False) -> list[dict[str, Any]]:
    print("\n=== Part 3: RGB VGG16 transfer learning ===")
    result_path = FIGURES_DIR / ("smoke_part3_rgb_vgg16_results.json" if smoke_test else "part3_rgb_vgg16_results.json")
    fig_path = FIGURES_DIR / ("smoke_part3_rgb_vgg16_freeze.png" if smoke_test else "part3_rgb_vgg16_freeze.png")
    ckpt_prefix = "smoke_part3_rgb_vgg16" if smoke_test else "part3_rgb_vgg16"
    if reuse_checkpoints and result_path.exists():
        results = read_json(result_path)
        if all(checkpoint_exists(r["checkpoint"]) for r in results):
            print(f"loading cached {result_path}")
            plot_4panel(results, [r["label"] for r in results], "Part 3 - RGB: VGG16 Freeze-Level Comparison", "Freeze option", fig_path)
            return results
    batch_size = 8 if smoke_test else 128
    epochs = 1 if smoke_test else 30
    min_epochs = 1 if smoke_test else 10
    patience = 1 if smoke_test else 6
    train_loader, val_loader, _, _ = make_loaders(
        RGB_MANIFEST,
        imagenet_train_transform(False),
        imagenet_val_transform(),
        batch_size=batch_size,
        smoke_test=smoke_test,
        smoke_per_class_train=6,
        smoke_per_class_val=4,
    )
    results: list[dict[str, Any]] = []
    for key, cfg in VGG_FREEZE_CONFIGS.items():
        label = key
        ckpt_path = CHECKPOINT_DIR / f"{ckpt_prefix}_{key}.pt"
        print(f"\n{key}: {cfg['description']}")
        model = VGG16Transfer(freeze_level=int(cfg["freeze_level"]))
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        history = train_image_classifier(
            model,
            train_loader,
            val_loader,
            ckpt_path,
            epochs=epochs,
            min_epochs=min_epochs,
            patience=patience,
            weight_decay=1e-5,
            param_groups=model.get_param_groups(lr_pretrained=1e-5, lr_head=3e-4),
        )
        results.append(
            {
                "label": label,
                "description": cfg["description"],
                "freeze_level": cfg["freeze_level"],
                "param_count": total,
                "trainable_params": trainable,
                "checkpoint": str(ckpt_path),
                "kind": "vgg16",
                **history,
            }
        )
        del model
        torch.cuda.empty_cache()
    write_json(result_path, results)
    plot_4panel(results, [r["label"] for r in results], "Part 3 - RGB: VGG16 Freeze-Level Comparison", "Freeze option", fig_path)
    return results


def run_part4_autoencoder(reuse_checkpoints: bool = True, smoke_test: bool = False) -> list[dict[str, Any]]:
    print("\n=== Part 4: IR autoencoder feature classifiers ===")
    result_path = FIGURES_DIR / ("smoke_part4_ir_ae_results.json" if smoke_test else "part4_ir_ae_results.json")
    fig_path = FIGURES_DIR / ("smoke_part4_ir_autoencoder_12_options.png" if smoke_test else "part4_ir_autoencoder_12_options.png")
    ckpt_prefix = "smoke_part4" if smoke_test else "part4"
    if reuse_checkpoints and result_path.exists():
        results = read_json(result_path)
        if all(checkpoint_exists(r["classifier_checkpoint"]) for r in results):
            print(f"loading cached {result_path}")
            plot_4panel(results, [r["label"] for r in results], "Part 4 - IR: AE Feature Classifier Options", "AE + regularization option", fig_path)
            return results
    batch_size = 32 if smoke_test else 512
    ae_epochs = 1 if smoke_test else 25
    ae_min_epochs = 1 if smoke_test else 8
    ae_patience = 1 if smoke_test else 5
    clf_epochs = 2 if smoke_test else 50
    clf_min_epochs = 1 if smoke_test else 15
    clf_patience = 1 if smoke_test else 8
    clf_batch_size = 32 if smoke_test else 1024
    train_loader, val_loader, _, _ = make_loaders(
        IR_MANIFEST,
        base_patch_transform(64, False),
        base_patch_transform(64, False),
        batch_size=batch_size,
        smoke_test=smoke_test,
        smoke_per_class_train=8,
        smoke_per_class_val=4,
    )
    results: list[dict[str, Any]] = []
    for ae_name, cfg in AE_CONFIGS.items():
        ae_ckpt = CHECKPOINT_DIR / f"{ckpt_prefix}_ir_{ae_name}_autoencoder.pt"
        print(f"\n{ae_name}: filters={cfg['filters']} bottleneck={cfg['bottleneck_dim']}")
        ae = ConvAutoEncoder(filters=cfg["filters"], bottleneck_dim=cfg["bottleneck_dim"])
        if reuse_checkpoints and checkpoint_exists(ae_ckpt):
            ae.load_state_dict(load_state_dict(ae_ckpt))
            ae_history = {"best_epoch": "loaded", "best_train_loss": None, "best_val_loss": None}
            print(f"  loaded {ae_ckpt}")
        else:
            ae_history = train_autoencoder(ae, train_loader, val_loader, ae_ckpt, ae_epochs, ae_min_epochs, ae_patience)
        train_features, train_labels = extract_features(ae, train_loader)
        val_features, val_labels = extract_features(ae, val_loader)
        train_ds = TensorDataset(train_features, train_labels)
        val_ds = TensorDataset(val_features, val_labels)
        for reg_name, reg_cfg in AE_REG_CONFIGS.items():
            label = f"{ae_name}-{reg_name}"
            clf_ckpt = CHECKPOINT_DIR / f"{ckpt_prefix}_ir_{label}_classifier.pt"
            print(f"  classifier {label}: dropout={reg_cfg['dropout']} wd={reg_cfg['weight_decay']}")
            classifier = DenseClassifier(input_dim=cfg["bottleneck_dim"], dropout=reg_cfg["dropout"])
            if reuse_checkpoints and checkpoint_exists(clf_ckpt):
                classifier.load_state_dict(load_state_dict(clf_ckpt))
                criterion = nn.CrossEntropyLoss()
                t_loader = DataLoader(train_ds, batch_size=clf_batch_size, shuffle=False)
                v_loader = DataLoader(val_ds, batch_size=clf_batch_size, shuffle=False)
                train_loss, train_acc, train_f1 = evaluate_feature_classifier(classifier.to(DEVICE), t_loader, criterion)
                val_loss, val_acc, val_f1 = evaluate_feature_classifier(classifier.to(DEVICE), v_loader, criterion)
                history = {
                    "best_epoch": "loaded",
                    "best_train_loss": train_loss,
                    "best_val_loss": val_loss,
                    "best_train_acc": train_acc,
                    "best_val_acc": val_acc,
                    "best_train_f1": train_f1,
                    "best_val_f1": val_f1,
                }
                print(f"    loaded {clf_ckpt}: val f1 {val_f1:.4f}")
            else:
                history = train_feature_classifier(
                    classifier,
                    train_ds,
                    val_ds,
                    clf_ckpt,
                    epochs=clf_epochs,
                    min_epochs=clf_min_epochs,
                    patience=clf_patience,
                    batch_size=clf_batch_size,
                    lr=1e-3,
                    weight_decay=float(reg_cfg["weight_decay"]),
                )
            results.append(
                {
                    "label": label,
                    "ae_config": ae_name,
                    "filters": cfg["filters"],
                    "bottleneck_dim": cfg["bottleneck_dim"],
                    "reg_config": reg_name,
                    "dropout": reg_cfg["dropout"],
                    "weight_decay": reg_cfg["weight_decay"],
                    "ae_checkpoint": str(ae_ckpt),
                    "classifier_checkpoint": str(clf_ckpt),
                    "kind": "ae_classifier",
                    "ae_history": ae_history,
                    **history,
                }
            )
            del classifier
            torch.cuda.empty_cache()
        del ae, train_features, val_features
        torch.cuda.empty_cache()
    write_json(result_path, results)
    plot_4panel(results, [r["label"] for r in results], "Part 4 - IR: AE Feature Classifier Options", "AE + regularization option", fig_path)
    return results


def run_part6_refinement(
    part3_results: list[dict[str, Any]],
    part4_results: list[dict[str, Any]],
    reuse_checkpoints: bool = True,
    smoke_test: bool = False,
) -> dict[str, Any]:
    print("\n=== Part 6: bounded refinement ===")
    result_path = FIGURES_DIR / ("smoke_part6_refinement_results.json" if smoke_test else "part6_refinement_results.json")
    if reuse_checkpoints and result_path.exists():
        print(f"loading cached {result_path}")
        return read_json(result_path)
    summary: dict[str, Any] = {"rgb_refined": None, "ir_refined": None, "notes": []}
    best_vgg = max(part3_results, key=lambda r: float(r["best_val_f1"]))
    if float(best_vgg["best_val_f1"]) >= RGB_PART1_BASELINE["best_val_f1"] - 0.01:
        print("RGB VGG is close to Part 1; running mild augmentation refinement.")
        ckpt_prefix = "smoke_part6" if smoke_test else "part6"
        ckpt_path = CHECKPOINT_DIR / f"{ckpt_prefix}_rgb_vgg_refined.pt"
        batch_size = 8 if smoke_test else 128
        epochs = 1 if smoke_test else 20
        train_loader, val_loader, _, _ = make_loaders(
            RGB_MANIFEST,
            imagenet_train_transform(True),
            imagenet_val_transform(),
            batch_size=batch_size,
            smoke_test=smoke_test,
            smoke_per_class_train=6,
            smoke_per_class_val=4,
        )
        model = VGG16Transfer(freeze_level=int(best_vgg["freeze_level"]))
        model.load_state_dict(load_state_dict(best_vgg["checkpoint"]))
        history = train_image_classifier(
            model,
            train_loader,
            val_loader,
            ckpt_path,
            epochs=epochs,
            min_epochs=1 if smoke_test else 8,
            patience=1 if smoke_test else 5,
            weight_decay=1e-5,
            param_groups=model.get_param_groups(lr_pretrained=5e-6, lr_head=1e-4),
        )
        summary["rgb_refined"] = {
            "label": "RGB VGG16 refined",
            "kind": "vgg16",
            "freeze_level": best_vgg["freeze_level"],
            "checkpoint": str(ckpt_path),
            **history,
        }
    else:
        summary["notes"].append("RGB VGG16 did not come within 0.01 F1 of RGB Part 1 Arch D; final RGB remains the custom CNN baseline unless report data says otherwise.")
    best_ae = max(part4_results, key=lambda r: float(r["best_val_f1"]))
    if float(best_ae["best_val_f1"]) >= IR_PART1_BASELINE["best_val_f1"] - 0.02:
        print("IR AE is close to Part 1; running classifier-only refinement.")
        ckpt_prefix = "smoke_part6" if smoke_test else "part6"
        ae_cfg = AE_CONFIGS[str(best_ae["ae_config"])]
        ae = ConvAutoEncoder(filters=ae_cfg["filters"], bottleneck_dim=ae_cfg["bottleneck_dim"])
        ae.load_state_dict(load_state_dict(best_ae["ae_checkpoint"]))
        train_loader, val_loader, _, _ = make_loaders(
            IR_MANIFEST,
            base_patch_transform(64, False),
            base_patch_transform(64, False),
            batch_size=32 if smoke_test else 512,
            smoke_test=smoke_test,
            smoke_per_class_train=8,
            smoke_per_class_val=4,
        )
        train_features, train_labels = extract_features(ae, train_loader)
        val_features, val_labels = extract_features(ae, val_loader)
        train_ds = TensorDataset(train_features, train_labels)
        val_ds = TensorDataset(val_features, val_labels)
        ckpt_path = CHECKPOINT_DIR / f"{ckpt_prefix}_ir_ae_refined_classifier.pt"
        classifier = DenseClassifier(input_dim=ae_cfg["bottleneck_dim"], dropout=float(best_ae["dropout"]))
        classifier.load_state_dict(load_state_dict(best_ae["classifier_checkpoint"]))
        history = train_feature_classifier(
            classifier,
            train_ds,
            val_ds,
            ckpt_path,
            epochs=2 if smoke_test else 30,
            min_epochs=1 if smoke_test else 10,
            patience=1 if smoke_test else 6,
            batch_size=32 if smoke_test else 1024,
            lr=5e-4,
            weight_decay=float(best_ae["weight_decay"]),
        )
        summary["ir_refined"] = {
            "label": "IR AE refined",
            "kind": "ae_classifier",
            "ae_config": best_ae["ae_config"],
            "filters": ae_cfg["filters"],
            "bottleneck_dim": ae_cfg["bottleneck_dim"],
            "dropout": best_ae["dropout"],
            "weight_decay": best_ae["weight_decay"],
            "ae_checkpoint": best_ae["ae_checkpoint"],
            "classifier_checkpoint": str(ckpt_path),
            **history,
        }
    else:
        summary["notes"].append("IR AE did not come within 0.02 F1 of IR Part 1 Arch D; final IR uses the stronger supervised CNN reference.")
    write_json(result_path, summary)
    return summary


def select_final_models(part3_results: list[dict[str, Any]], part4_results: list[dict[str, Any]], refinement: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any]]:
    rgb_candidates: list[dict[str, Any]] = [RGB_PART1_BASELINE]
    rgb_candidates.extend(
        {
            **r,
            "label": f"RGB VGG16 {r['label']}",
            "modality": "RGB",
            "source": f"part3_{r['label']}",
        }
        for r in part3_results
    )
    if refinement.get("rgb_refined"):
        rgb_candidates.append({**refinement["rgb_refined"], "modality": "RGB", "source": "part6_rgb_refined"})
    ir_candidates: list[dict[str, Any]] = [IR_PART1_BASELINE]
    if checkpoint_exists(IR_REGULARIZED_REFERENCE["checkpoint"]):
        ir_candidates.append(IR_REGULARIZED_REFERENCE)
    ir_candidates.extend(
        {
            **r,
            "label": f"IR {r['label']}",
            "modality": "IR",
            "source": f"part4_{r['label']}",
        }
        for r in part4_results
    )
    if refinement.get("ir_refined"):
        ir_candidates.append({**refinement["ir_refined"], "modality": "IR", "source": "part6_ir_refined"})
    final_rgb = max(rgb_candidates, key=lambda r: float(r.get("best_val_f1", 0.0)))
    final_ir = max(ir_candidates, key=lambda r: float(r.get("best_val_f1", 0.0)))
    final_summary = {"final_rgb": final_rgb, "final_ir": final_ir, "rgb_candidates": rgb_candidates, "ir_candidates": ir_candidates}
    write_json(FIGURES_DIR / "report2_final_model_selection.json", final_summary)
    print(f"Final RGB: {final_rgb['label']} F1={float(final_rgb['best_val_f1']):.4f}")
    print(f"Final IR: {final_ir['label']} F1={float(final_ir['best_val_f1']):.4f}")
    return final_rgb, final_ir


def load_prediction_model(model_info: dict[str, Any]) -> tuple[str, nn.Module | tuple[ConvAutoEncoder, DenseClassifier], transforms.Compose, int]:
    kind = model_info.get("kind")
    if kind == "vgg16":
        model = VGG16Transfer(freeze_level=int(model_info.get("freeze_level", 1)))
        model.load_state_dict(load_state_dict(model_info["checkpoint"]))
        return "image", model, imagenet_val_transform(), 128
    if kind == "ae_classifier":
        ae_cfg = AE_CONFIGS[str(model_info["ae_config"])]
        ae = ConvAutoEncoder(filters=ae_cfg["filters"], bottleneck_dim=ae_cfg["bottleneck_dim"])
        ae.load_state_dict(load_state_dict(model_info["ae_checkpoint"]))
        clf = DenseClassifier(input_dim=ae_cfg["bottleneck_dim"], dropout=float(model_info.get("dropout", 0.2)))
        clf.load_state_dict(load_state_dict(model_info["classifier_checkpoint"]))
        return "ae", (ae, clf), base_patch_transform(64, False), 512
    model = CNNArchD()
    model.load_state_dict(load_state_dict(model_info["checkpoint"]))
    return "image", model, base_patch_transform(64, False), 512


def run_prediction_for_model(model_info: dict[str, Any], manifest_path: Path, output_csv: Path, smoke_test: bool = False) -> pd.DataFrame:
    mode, model_obj, transform, batch_size = load_prediction_model(model_info)
    _, val_loader, _, val_df = make_loaders(
        manifest_path,
        transform,
        transform,
        batch_size=batch_size if not smoke_test else 32,
        smoke_test=smoke_test,
        smoke_per_class_train=1,
        smoke_per_class_val=4,
        shuffle_train=False,
    )
    if mode == "image":
        pred_df = predict_image_classifier(model_obj, val_loader, val_df)  # type: ignore[arg-type]
    else:
        ae, clf = model_obj  # type: ignore[misc]
        features, labels = extract_features(ae, val_loader)
        pred_df = predict_feature_classifier(clf, features, labels, val_df)
    output_csv.parent.mkdir(exist_ok=True)
    pred_df.to_csv(output_csv, index=False)
    return pred_df


def select_misclassified_items(pred_df: pd.DataFrame, n_per_class: int = 2) -> tuple[list[dict[str, Any]], dict[str, int]]:
    items: list[dict[str, Any]] = []
    counts: dict[str, int] = {}
    for class_name in CLASSES:
        misses = pred_df[(pred_df["true_class"] == class_name) & (~pred_df["correct"])].copy()
        counts[class_name] = len(misses)
        misses = misses.sort_values("confidence", ascending=False).head(n_per_class)
        for _, row in misses.iterrows():
            items.append(row.to_dict())
    return items, counts


def load_rgb_to_thermal_map() -> dict[str, str]:
    with INDEX_JSON.open(encoding="utf-8") as f:
        index = json.load(f)
    thermal_to_rgb: dict[str, str] = {}
    for video in index.get("videos", []):
        desc = video.get("description", "")
        if '"RGB"' not in desc:
            continue
        start = desc.find('"RGB"')
        frag = desc[start:]
        left = frag.find('"', frag.find(":"))
        right = frag.find('"', left + 1)
        if left >= 0 and right > left:
            thermal_to_rgb[str(video["id"])] = frag[left + 1 : right]
    return {rgb: thermal for thermal, rgb in thermal_to_rgb.items()}


def find_sensor_pairs(rgb_pred: pd.DataFrame, ir_pred: pd.DataFrame, want_rgb_correct: bool) -> list[dict[str, Any]]:
    paired = pd.read_csv(PAIRED_SCENES)
    rgb_to_thermal = load_rgb_to_thermal_map()
    out: list[dict[str, Any]] = []
    for class_name in CLASSES:
        found = None
        for _, scene in paired.iterrows():
            rgb_vid = scene["rgb_video_id"]
            ir_vid = scene["thermal_video_id"]
            frame = int(scene["frame_index"])
            rgb_rows = rgb_pred[
                (rgb_pred["video_id"] == rgb_vid)
                & (rgb_pred["frame_index"].astype(int) == frame)
                & (rgb_pred["true_class"] == class_name)
                & (rgb_pred["correct"] == want_rgb_correct)
            ]
            ir_rows = ir_pred[
                (ir_pred["video_id"] == ir_vid)
                & (ir_pred["frame_index"].astype(int) == frame)
                & (ir_pred["true_class"] == class_name)
                & (ir_pred["correct"] != want_rgb_correct)
            ]
            if len(rgb_rows) and len(ir_rows):
                found = {"rgb": rgb_rows.iloc[0].to_dict(), "ir": ir_rows.iloc[0].to_dict(), "class_name": class_name, "match": "same class/frame"}
                break
        if found:
            out.append(found)
        if len(out) >= 4:
            break
    if out:
        return out
    for _, scene in paired.iterrows():
        rgb_vid = scene["rgb_video_id"]
        ir_vid = scene["thermal_video_id"]
        frame = int(scene["frame_index"])
        rgb_rows = rgb_pred[
            (rgb_pred["video_id"] == rgb_vid)
            & (rgb_pred["frame_index"].astype(int) == frame)
            & (rgb_pred["correct"] == want_rgb_correct)
        ]
        ir_rows = ir_pred[
            (ir_pred["video_id"] == ir_vid)
            & (ir_pred["frame_index"].astype(int) == frame)
            & (ir_pred["correct"] != want_rgb_correct)
        ]
        if len(rgb_rows) and len(ir_rows):
            out.append({"rgb": rgb_rows.iloc[0].to_dict(), "ir": ir_rows.iloc[0].to_dict(), "class_name": "scene", "match": "same paired frame"})
        if len(out) >= 4:
            break
    return out


def run_part5_analysis(final_rgb: dict[str, Any], final_ir: dict[str, Any], smoke_test: bool = False) -> dict[str, Any]:
    print("\n=== Part 5: error and sensor analysis ===")
    prefix = "smoke_" if smoke_test else ""
    rgb_csv = FIGURES_DIR / f"{prefix}part5_rgb_predictions.csv"
    ir_csv = FIGURES_DIR / f"{prefix}part5_ir_predictions.csv"
    rgb_pred = run_prediction_for_model(final_rgb, RGB_MANIFEST, rgb_csv, smoke_test)
    ir_pred = run_prediction_for_model(final_ir, IR_MANIFEST, ir_csv, smoke_test)
    rgb_items, rgb_counts = select_misclassified_items(rgb_pred, 2)
    ir_items, ir_counts = select_misclassified_items(ir_pred, 2)
    rgb_grid = FIGURES_DIR / f"{prefix}part5_rgb_misclassified_by_class.png"
    ir_grid = FIGURES_DIR / f"{prefix}part5_ir_misclassified_by_class.png"
    plot_misclassified_grid(rgb_items, "RGB: Up to 2 Misclassified Validation Samples Per Class", rgb_grid)
    plot_misclassified_grid(ir_items, "IR: Up to 2 Misclassified Validation Samples Per Class", ir_grid)
    rgb_correct_ir_wrong = find_sensor_pairs(rgb_pred, ir_pred, want_rgb_correct=True)
    ir_correct_rgb_wrong = find_sensor_pairs(rgb_pred, ir_pred, want_rgb_correct=False)
    rgb_ir_fig = FIGURES_DIR / f"{prefix}part5_rgb_correct_ir_wrong.png"
    ir_rgb_fig = FIGURES_DIR / f"{prefix}part5_ir_correct_rgb_wrong.png"
    plot_sensor_pairs(rgb_correct_ir_wrong, "Paired Scenes: RGB Correct, IR Incorrect", rgb_ir_fig)
    plot_sensor_pairs(ir_correct_rgb_wrong, "Paired Scenes: IR Correct, RGB Incorrect", ir_rgb_fig)
    summary = {
        "rgb_prediction_csv": str(rgb_csv),
        "ir_prediction_csv": str(ir_csv),
        "rgb_misclassified_counts": rgb_counts,
        "ir_misclassified_counts": ir_counts,
        "rgb_misclassified_figure": str(rgb_grid),
        "ir_misclassified_figure": str(ir_grid),
        "rgb_correct_ir_wrong_count": len(rgb_correct_ir_wrong),
        "ir_correct_rgb_wrong_count": len(ir_correct_rgb_wrong),
        "rgb_correct_ir_wrong_figure": str(rgb_ir_fig),
        "ir_correct_rgb_wrong_figure": str(ir_rgb_fig),
        "rgb_val_acc": float(rgb_pred["correct"].mean()),
        "ir_val_acc": float(ir_pred["correct"].mean()),
    }
    write_json(FIGURES_DIR / f"{prefix}part5_analysis_summary.json", summary)
    return summary


def draw_header(c: canvas.Canvas, title: str, page_no: int) -> None:
    width, height = letter
    c.setFillColor(colors.HexColor("#1f2937"))
    c.setFont("Helvetica-Bold", 9)
    c.drawString(0.45 * inch, height - 0.35 * inch, "CSE 547 Final Project Report 2")
    c.drawRightString(width - 0.45 * inch, height - 0.35 * inch, f"Jose Fuentes | Page {page_no}")
    c.setFont("Helvetica-Bold", 15)
    c.drawString(0.45 * inch, height - 0.65 * inch, title)
    c.setStrokeColor(colors.HexColor("#9ca3af"))
    c.line(0.45 * inch, height - 0.78 * inch, width - 0.45 * inch, height - 0.78 * inch)


def draw_wrapped(c: canvas.Canvas, text: str, x: float, y: float, width: float, font: str = "Helvetica", size: int = 8.5, leading: float = 11) -> float:
    c.setFont(font, size)
    max_chars = max(20, int(width / (size * 0.48)))
    lines: list[str] = []
    for para in text.split("\n"):
        lines.extend(textwrap.wrap(para, max_chars) or [""])
    for line in lines:
        c.drawString(x, y, line)
        y -= leading
    return y


def draw_table(c: canvas.Canvas, rows: list[list[str]], x: float, y_top: float, col_widths: list[float], row_h: float = 0.22 * inch, font_size: int = 7.2) -> float:
    y = y_top
    for r, row in enumerate(rows):
        x_pos = x
        for col, val in enumerate(row):
            c.setFillColor(colors.HexColor("#e5f0fb") if r == 0 else colors.white)
            c.rect(x_pos, y - row_h, col_widths[col], row_h, stroke=1, fill=1)
            c.setFillColor(colors.black)
            c.setFont("Helvetica-Bold" if r == 0 else "Helvetica", font_size)
            c.drawString(x_pos + 3, y - row_h + 4, str(val)[:40])
            x_pos += col_widths[col]
        y -= row_h
    return y


def draw_image_fit(c: canvas.Canvas, path: Path, x: float, y: float, w: float, h: float) -> None:
    if not path.exists():
        c.setStrokeColor(colors.red)
        c.rect(x, y, w, h, stroke=1, fill=0)
        c.setFillColor(colors.red)
        c.drawCentredString(x + w / 2, y + h / 2, f"Missing: {path.name}")
        return
    try:
        img = ImageReader(str(path))
        iw, ih = img.getSize()
        scale = min(w / iw, h / ih)
        dw = iw * scale
        dh = ih * scale
        c.drawImage(img, x + (w - dw) / 2, y + (h - dh) / 2, width=dw, height=dh, preserveAspectRatio=True, mask="auto")
    except Exception:
        c.rect(x, y, w, h, stroke=1, fill=0)
        c.drawCentredString(x + w / 2, y + h / 2, f"Could not draw {path.name}")


def metric_row(label: str, result: dict[str, Any]) -> list[str]:
    return [
        label,
        f"{float(result.get('best_train_acc', 0)):.4f}",
        f"{float(result.get('best_val_acc', 0)):.4f}",
        f"{float(result.get('best_train_f1', 0)):.4f}",
        f"{float(result.get('best_val_f1', 0)):.4f}",
    ]


def build_report_pdf(
    part3_results: list[dict[str, Any]],
    part4_results: list[dict[str, Any]],
    refinement: dict[str, Any],
    final_rgb: dict[str, Any],
    final_ir: dict[str, Any],
    part5_summary: dict[str, Any],
    smoke_test: bool = False,
) -> None:
    print("\n=== Building final PDF report ===")
    prefix = "smoke_" if smoke_test else ""
    c = canvas.Canvas(str(REPORT_PDF), pagesize=letter)
    width, height = letter
    margin = 0.45 * inch

    draw_header(c, "Objective, Baselines, and RGB Transfer Learning", 1)
    y = height - 0.98 * inch
    intro = (
        "This report completes the final project requirements after Report 1. The task is eight-class object "
        "recognition for autonomous driving using RGB and infrared patches. Report 1 established the custom CNN "
        "baselines; this report adds RGB VGG16 transfer learning, IR autoencoder features, sensor comparison, "
        "misclassification analysis, and final model selection."
    )
    y = draw_wrapped(c, intro, margin, y, width - 2 * margin, size=8.2, leading=10)
    y -= 0.08 * inch
    baseline_rows = [
        ["Model", "Train Acc", "Val Acc", "Train F1", "Val F1"],
        metric_row("RGB Part 1 Arch D", RGB_PART1_BASELINE),
        metric_row("IR Part 1 Arch D", IR_PART1_BASELINE),
    ]
    draw_table(c, baseline_rows, margin, y, [1.8 * inch, 1.0 * inch, 1.0 * inch, 1.0 * inch, 1.0 * inch])
    best_vgg = max(part3_results, key=lambda r: float(r["best_val_f1"]))
    vgg_rows = [["Freeze", "Train Acc", "Val Acc", "Train F1", "Val F1"]] + [metric_row(r["label"], r) for r in part3_results]
    draw_table(c, vgg_rows, margin, y - 0.9 * inch, [0.9 * inch, 1.0 * inch, 1.0 * inch, 1.0 * inch, 1.0 * inch])
    draw_image_fit(c, FIGURES_DIR / f"{prefix}part3_rgb_vgg16_freeze.png", margin, 0.65 * inch, width - 2 * margin, 4.15 * inch)
    c.showPage()

    draw_header(c, "IR Autoencoder Feature Classifiers", 2)
    best_ae = max(part4_results, key=lambda r: float(r["best_val_f1"]))
    y = height - 1.0 * inch
    ae_text = (
        f"The IR autoencoder experiment used six encoder configurations and two dense-head regularization settings, "
        f"for 12 total options. The best AE option was {best_ae['label']} with validation F1 "
        f"{float(best_ae['best_val_f1']):.4f}. The supervised IR Part 1 baseline validation F1 was "
        f"{IR_PART1_BASELINE['best_val_f1']:.4f}."
    )
    draw_wrapped(c, ae_text, margin, y, width - 2 * margin, size=8.5, leading=10.5)
    draw_image_fit(c, FIGURES_DIR / f"{prefix}part4_ir_autoencoder_12_options.png", margin, 0.7 * inch, width - 2 * margin, 6.6 * inch)
    c.showPage()

    draw_header(c, "Final Model Selection and Refinement", 3)
    final_rows = [
        ["Candidate", "Train Acc", "Val Acc", "Train F1", "Val F1"],
        metric_row("RGB final", final_rgb),
        metric_row("Best RGB VGG16", best_vgg),
        metric_row("RGB Part 1 Arch D", RGB_PART1_BASELINE),
        metric_row("IR final", final_ir),
        metric_row("Best IR AE", best_ae),
        metric_row("IR Part 1 Arch D", IR_PART1_BASELINE),
    ]
    draw_table(c, final_rows, margin, height - 1.05 * inch, [1.65 * inch, 1.0 * inch, 1.0 * inch, 1.0 * inch, 1.0 * inch])
    notes = [
        f"Final RGB model: {final_rgb['label']} (validation F1 {float(final_rgb['best_val_f1']):.4f}).",
        f"Final IR model: {final_ir['label']} (validation F1 {float(final_ir['best_val_f1']):.4f}).",
        "Blind-test strategy: select by validation weighted F1 and avoid using withheld labels. Transfer learning and AE results are retained even when the custom CNN remains stronger.",
    ]
    if refinement.get("notes"):
        notes.extend(refinement["notes"])
    y = height - 2.9 * inch
    for note in notes:
        y = draw_wrapped(c, "- " + note, margin, y, width - 2 * margin, size=9, leading=12)
        y -= 0.04 * inch
    comparison = (
        "Observed reliability pattern: RGB generally benefits from color and texture detail for traffic signs, lights, "
        "and object boundaries. Infrared can be more stable where visible color/texture is degraded, but small or low-resolution "
        "classes remain difficult in both sensors. The paired examples on page 5 show cases where the sensors disagree."
    )
    draw_wrapped(c, comparison, margin, y - 0.15 * inch, width - 2 * margin, size=9, leading=12)
    c.showPage()

    draw_header(c, "Misclassified Samples By Sensor", 4)
    y = height - 0.98 * inch
    miss_text = (
        "The grids below show up to two validation misclassifications per class where available. Counts are tracked in "
        "the generated Part 5 JSON summary so classes with fewer than two misses are explicitly documented."
    )
    draw_wrapped(c, miss_text, margin, y, width - 2 * margin, size=8.3, leading=10)
    draw_image_fit(c, FIGURES_DIR / f"{prefix}part5_rgb_misclassified_by_class.png", margin, 4.95 * inch, width - 2 * margin, 2.8 * inch)
    draw_image_fit(c, FIGURES_DIR / f"{prefix}part5_ir_misclassified_by_class.png", margin, 0.55 * inch, width - 2 * margin, 4.05 * inch)
    c.showPage()

    draw_header(c, "RGB-vs-IR Sensor Disagreement Analysis", 5)
    draw_image_fit(c, FIGURES_DIR / f"{prefix}part5_rgb_correct_ir_wrong.png", margin, 4.95 * inch, width - 2 * margin, 3.0 * inch)
    draw_image_fit(c, FIGURES_DIR / f"{prefix}part5_ir_correct_rgb_wrong.png", margin, 1.35 * inch, width - 2 * margin, 3.1 * inch)
    bottom_text = (
        f"Paired examples found: RGB-correct/IR-wrong = {part5_summary.get('rgb_correct_ir_wrong_count', 0)}, "
        f"IR-correct/RGB-wrong = {part5_summary.get('ir_correct_rgb_wrong_count', 0)}. "
        "When exact same-class paired examples were unavailable, the notebook used same-frame paired-scene examples and labels them as scene-level comparisons."
    )
    draw_wrapped(c, bottom_text, margin, 1.03 * inch, width - 2 * margin, size=8.5, leading=10.5)
    c.save()
    print(f"saved {REPORT_PDF}")


def write_video_outline(final_rgb: dict[str, Any], final_ir: dict[str, Any]) -> None:
    text = f"""# Report 2 Video Outline

## First 5 minutes: notebook and code
1. Open `CSE547_FinalProject_Report2_Fuentes.ipynb`.
2. Show setup: seeds, manifests, device, checkpoint reuse, and class map.
3. Explain Part 3 VGG16: pretrained conv base, three freeze settings, two dense layers.
4. Explain Part 4 AE: six encoder configs, frozen features, two dense regularization heads.
5. Show generated artifacts in `figures/` and `checkpoints/`.

## Second 5 minutes: report and results
1. Open `CSE547_FinalProject_Report2_Fuentes.pdf`.
2. Compare Part 3 VGG16 results to RGB Part 1 Arch D.
3. Compare Part 4 AE results to IR Part 1 Arch D.
4. Discuss final model choices:
   - RGB: {final_rgb['label']} with validation F1 {float(final_rgb['best_val_f1']):.4f}
   - IR: {final_ir['label']} with validation F1 {float(final_ir['best_val_f1']):.4f}
5. Explain misclassified samples and paired RGB/IR disagreement examples.
6. Close with blind-test strategy: use validation-selected best models only, no withheld labels.
"""
    VIDEO_OUTLINE.write_text(text, encoding="utf-8")
    print(f"saved {VIDEO_OUTLINE}")


def write_final_notebook() -> None:
    if "__file__" not in globals():
        print("Notebook context detected; skipping notebook self-regeneration.")
        return
    source = Path(__file__).read_text(encoding="utf-8")
    source_without_cli = source.split("\ndef parse_args(", 1)[0]
    cells = [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# CSE 547 Final Project Report 2 - Source Notebook\n",
                "\n",
                "This notebook is self-contained: it defines the models, training loops, plotting, prediction mining, report generation, and video outline generation used for the final submission.\n",
            ],
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["REUSE_CHECKPOINTS = True\n", "SMOKE_TEST = False\n"],
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## Implementation\n", "Run this cell to define all project code.\n"],
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": source_without_cli.splitlines(keepends=True),
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## Run Complete Pipeline\n"],
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "outputs = run_all(reuse_checkpoints=REUSE_CHECKPOINTS, smoke_test=SMOKE_TEST)\n",
                "outputs\n",
            ],
        },
    ]
    nb = {
        "cells": cells,
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "pygments_lexer": "ipython3"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    FINAL_NOTEBOOK.write_text(json.dumps(nb, indent=1), encoding="utf-8")
    print(f"saved {FINAL_NOTEBOOK}")


def verify_outputs(smoke_test: bool = False) -> list[str]:
    prefix = "smoke_" if smoke_test else ""
    required = [
        FIGURES_DIR / f"{prefix}part3_rgb_vgg16_freeze.png",
        FIGURES_DIR / f"{prefix}part3_rgb_vgg16_results.json",
        FIGURES_DIR / f"{prefix}part4_ir_autoencoder_12_options.png",
        FIGURES_DIR / f"{prefix}part4_ir_ae_results.json",
        FIGURES_DIR / f"{prefix}part5_rgb_misclassified_by_class.png",
        FIGURES_DIR / f"{prefix}part5_ir_misclassified_by_class.png",
        FIGURES_DIR / f"{prefix}part5_rgb_correct_ir_wrong.png",
        FIGURES_DIR / f"{prefix}part5_ir_correct_rgb_wrong.png",
        REPORT_PDF,
        FINAL_NOTEBOOK,
        VIDEO_OUTLINE,
    ]
    missing = [str(p) for p in required if not p.exists() or p.stat().st_size == 0]
    if missing:
        print("Missing outputs:")
        for p in missing:
            print(f"  {p}")
    else:
        print("All required outputs are present.")
    return missing


def run_all(reuse_checkpoints: bool = True, smoke_test: bool = False, skip_training: bool = False) -> dict[str, Any]:
    set_seed()
    ensure_dirs()
    print(f"Device: {DEVICE}")
    if skip_training:
        part3 = read_json(FIGURES_DIR / "part3_rgb_vgg16_results.json")
        part4 = read_json(FIGURES_DIR / "part4_ir_ae_results.json")
    else:
        part3 = run_part3_vgg16(reuse_checkpoints=reuse_checkpoints, smoke_test=smoke_test)
        part4 = run_part4_autoencoder(reuse_checkpoints=reuse_checkpoints, smoke_test=smoke_test)
    refinement = run_part6_refinement(part3, part4, reuse_checkpoints=reuse_checkpoints, smoke_test=smoke_test)
    final_rgb, final_ir = select_final_models(part3, part4, refinement)
    part5 = run_part5_analysis(final_rgb, final_ir, smoke_test=smoke_test)
    build_report_pdf(part3, part4, refinement, final_rgb, final_ir, part5, smoke_test=smoke_test)
    write_video_outline(final_rgb, final_ir)
    if "__file__" in globals():
        write_final_notebook()
    else:
        print("Notebook context detected; preserving current notebook file.")
    missing = verify_outputs(smoke_test=smoke_test)
    return {
        "part3": part3,
        "part4": part4,
        "refinement": refinement,
        "final_rgb": final_rgb,
        "final_ir": final_ir,
        "part5": part5,
        "missing": missing,
    }



## Run Complete Pipeline


In [ ]:
outputs = run_all(reuse_checkpoints=REUSE_CHECKPOINTS, smoke_test=SMOKE_TEST)
outputs
